# Notebook 3: Inter-Event Time Regime Classification


In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, ".")
from src.catalog import estimate_mc
from src.interevent import compute_interevent_times, fit_distributions, classify_regime
from src.spatial import assign_cells
from src.gutenberg_richter import compute_bvalue_grid
from src.plotting import setup_style, save_figure, plot_regime_map
from src.external_data import (
    load_gsrm_strain, resample_strain_to_grid,
    load_pb2002_boundaries, classify_grid_tectonic_settings
)

setup_style()

In [2]:
df = pd.read_csv("data/earthquake_catalog_global.csv")
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
print(f"Loaded {len(df):,} events")


Loaded 681,450 events


## 3.1 Spatial Gridding and IET Computation


In [3]:
df_cells = assign_cells(df, cell_size=2.0)

results = []
example_cells = {"poisson": None, "clustered": None, "quasi_periodic": None, "ambiguous": None}

for (clat, clon), grp in df_cells.groupby(["cell_lat", "cell_lon"]):
    if len(grp) < 100:
        continue

    times_sorted = grp.sort_values("time")["time"].values
    iet = compute_interevent_times(times_sorted)
    iet_hours = iet / 3600.0  # Convert to hours
    iet_hours = iet_hours[iet_hours > 0]

    if len(iet_hours) < 50:
        continue

    median_iet_days = np.median(iet_hours) / 24.0
    if median_iet_days > 30:
        continue

    try:
        fit_result = fit_distributions(iet_hours)
        regime = classify_regime(fit_result)
    except Exception:
        continue

    # Get Weibull k if available
    weibull_k = fit_result["weibull"]["params"][0]

    results.append({
        "cell_lat": clat,
        "cell_lon": clon,
        "regime": regime,
        "best_aic": fit_result["best_aic"],
        "weibull_k": weibull_k,
        "n_events": len(grp),
        "median_iet_hours": np.median(iet_hours),
    })

    # Save example cells
    if example_cells[regime] is None:
        example_cells[regime] = {"iet_hours": iet_hours, "fit_result": fit_result,
                                  "lat": clat, "lon": clon, "n": len(grp)}

regime_df = pd.DataFrame(results)
print(f"Classified {len(regime_df)} cells")
print(regime_df["regime"].value_counts())


Classified 715 cells
regime
clustered    650
ambiguous     53
poisson       12
Name: count, dtype: int64


## 3.2 Global Regime Map


In [4]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_regime_map(regime_df["cell_lat"], regime_df["cell_lon"], regime_df["regime"], ax=ax)
save_figure(fig, "03_regime_classification_map")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_50119/3266278282.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.3 Weibull k Map


In [5]:
from src.plotting import plot_global_map

fig, ax = plt.subplots(figsize=(14, 7))
plot_global_map(
    regime_df["cell_lat"], regime_df["cell_lon"], regime_df["weibull_k"],
    cmap="coolwarm", vmin=0.5, vmax=1.5,
    label="Weibull shape parameter k",
    title="Weibull k Parameter (k<1: clustered, k=1: Poisson, k>1: quasi-periodic)",
    ax=ax
)
save_figure(fig, "03_weibull_k_map")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_50119/936925768.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.4 Example Distributions


In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes_flat = axes.flatten()

for i, (regime_name, label) in enumerate([
    ("poisson", "Poisson-like"), ("clustered", "Clustered-like"),
    ("quasi_periodic", "Quasi-periodic-like"), ("ambiguous", "Ambiguous")
]):
    ax = axes_flat[i]
    ex = example_cells.get(regime_name)
    if ex is None:
        ax.set_title(f"{label} — no example found")
        continue

    iet = ex["iet_hours"]
    fit = ex["fit_result"]

    # Empirical CDF
    sorted_iet = np.sort(iet)
    ecdf = np.arange(1, len(sorted_iet) + 1) / len(sorted_iet)
    ax.plot(sorted_iet, ecdf, "k-", linewidth=1.5, label="Empirical CDF")

    # Fitted CDFs
    x = np.linspace(sorted_iet[0], np.percentile(sorted_iet, 99), 200)
    colors = {"exponential": "#457B9D", "gamma": "#E63946",
              "lognormal": "#F4A261", "weibull": "#2A9D8F"}
    for dist_name, color in colors.items():
        params = fit[dist_name]["params"]
        dist = getattr(stats, "expon" if dist_name == "exponential" else
                       "lognorm" if dist_name == "lognormal" else
                       "weibull_min" if dist_name == "weibull" else "gamma")
        cdf_vals = dist.cdf(x, *params)
        aic = fit[dist_name]["aic"]
        ax.plot(x, cdf_vals, color=color, linewidth=1, alpha=0.8,
                label=f"{dist_name} (AIC={aic:.0f})")

    ax.set_xlabel("Inter-event time (hours)")
    ax.set_ylabel("CDF")
    ax.set_title(f"{label} (lat={ex['lat']:.0f}, lon={ex['lon']:.0f})")
    ax.legend(fontsize=7)
    ax.set_xscale("log")

fig.suptitle("Example IET Distributions by Regime", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "03_example_distributions")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_50119/2259060495.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.5 Weibull k vs. b-value


In [7]:
bvalue_grid = compute_bvalue_grid(df, cell_size=2.0, min_events=50)
merged = regime_df.merge(bvalue_grid[["cell_lat", "cell_lon", "b_value"]],
                         on=["cell_lat", "cell_lon"])

fig, ax = plt.subplots(figsize=(10, 7))
color_map = {"poisson": "#457B9D", "clustered": "#E63946",
             "quasi_periodic": "#2A9D8F", "ambiguous": "#AAAAAA"}
for regime_name in color_map:
    subset = merged[merged["regime"] == regime_name]
    ax.scatter(subset["b_value"], subset["weibull_k"], c=color_map[regime_name],
               s=20, alpha=0.5, label=regime_name.replace("_", " ").title())

ax.set_xlabel("b-value")
ax.set_ylabel("Weibull k")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Weibull Shape Parameter vs. b-value")
ax.legend()
save_figure(fig, "03_k_vs_bvalue")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_50119/3514603236.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.6 Weibull k vs. Depth


In [8]:
df_cells_depth = assign_cells(df, cell_size=2.0)
depth_med = df_cells_depth.groupby(["cell_lat", "cell_lon"])["depth"].median().reset_index()
depth_med.columns = ["cell_lat", "cell_lon", "median_depth"]

k_depth = regime_df.merge(depth_med, on=["cell_lat", "cell_lon"])

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(k_depth["median_depth"], k_depth["weibull_k"],
                c=k_depth["n_events"], cmap="viridis", s=15, alpha=0.5,
                norm=plt.matplotlib.colors.LogNorm())
plt.colorbar(sc, ax=ax, label="Number of events")
ax.set_xlabel("Median Depth (km)")
ax.set_ylabel("Weibull k")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Weibull k vs. Predominant Event Depth")
save_figure(fig, "03_k_vs_depth")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_50119/3159379449.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.7 Regime vs. Geodetic Strain Rate

Do faster-deforming regions have different temporal clustering patterns? We join the regime classification with the GSRM v2.1 strain rate model to test whether strain rate predicts clustering behavior.

In [9]:
# Load and resample GSRM strain rates
strain_raw = load_gsrm_strain()
strain_grid = resample_strain_to_grid(strain_raw, grid_size=2.0)

# Join with regime classification
regime_strain = regime_df.merge(
    strain_grid, left_on=["cell_lat", "cell_lon"], right_on=["lat_bin", "lon_bin"], how="inner"
)
regime_strain = regime_strain[regime_strain["mean_second_invariant"] > 0]
print(f"Matched cells (regime + strain): {len(regime_strain)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Strain rate distribution by regime
ax = axes[0]
color_map = {"poisson": "#457B9D", "clustered": "#E63946",
             "quasi_periodic": "#2A9D8F", "ambiguous": "#AAAAAA"}
for regime_name in ["poisson", "clustered", "ambiguous"]:
    subset = regime_strain[regime_strain["regime"] == regime_name]
    if len(subset) > 0:
        ax.hist(np.log10(subset["mean_second_invariant"]), bins=20, alpha=0.5,
                color=color_map[regime_name], label=f"{regime_name} (n={len(subset)})")
ax.set_xlabel("log₁₀ Strain Rate (nanostr/yr)")
ax.set_ylabel("Count")
ax.set_title("Strain Rate Distribution by Regime")
ax.legend()

# Panel 2: Weibull k vs strain rate
ax = axes[1]
from scipy.stats import spearmanr
sc = ax.scatter(regime_strain["mean_second_invariant"], regime_strain["weibull_k"],
                c=[color_map[r] for r in regime_strain["regime"]], s=15, alpha=0.5)
ax.set_xscale("log")
ax.set_xlabel("Geodetic Strain Rate (nanostr/yr)")
ax.set_ylabel("Weibull k")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Temporal Clustering vs. Strain Rate")

rho, p = spearmanr(np.log10(regime_strain["mean_second_invariant"]), regime_strain["weibull_k"])
ax.text(0.05, 0.95, f"Spearman ρ = {rho:.3f}\np = {p:.2e}",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

fig.tight_layout()
save_figure(fig, "03_regime_vs_strain_rate")
plt.show()

# Median strain rate by regime
print("\nMedian strain rate by regime:")
for regime_name in ["poisson", "clustered", "ambiguous"]:
    subset = regime_strain[regime_strain["regime"] == regime_name]
    if len(subset) > 0:
        print(f"  {regime_name:15s}: {subset['mean_second_invariant'].median():.1f} nanostr/yr (n={len(subset)})")

Matched cells (regime + strain): 669


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_50119/1585445899.py:44: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  fig.tight_layout()
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_50119/1585445899.py:44: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.tight_layout()


/Users/christopherortiz/Desktop/memory-of-the-earth/src/plotting.py:87: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  fig.savefig(path, dpi=DEFAULT_DPI, bbox_inches="tight")
/Users/christopherortiz/Desktop/memory-of-the-earth/src/plotting.py:87: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.savefig(path, dpi=DEFAULT_DPI, bbox_inches="tight")



Median strain rate by regime:
  poisson        : 27.0 nanostr/yr (n=11)
  clustered      : 129.6 nanostr/yr (n=611)
  ambiguous      : 55.4 nanostr/yr (n=47)


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_50119/1585445899.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.8 Regime vs. Tectonic Setting (PB2002)

Using the PB2002 plate boundary model, we classify each grid cell by tectonic setting and test whether the IET regime distribution varies systematically. The key question: are subduction zones more clustered than intraplate regions?

In [10]:
# Classify regime grid cells by tectonic setting
boundaries = load_pb2002_boundaries()
tec_settings = classify_grid_tectonic_settings(
    regime_df["cell_lat"].values, regime_df["cell_lon"].values,
    boundaries, radius_deg=2.0
)
regime_df["tectonic_setting"] = tec_settings

setting_order = ["subduction", "convergent", "transform", "spreading", "rift", "intraplate"]
setting_colors = {
    "subduction": "#E63946", "convergent": "#F4A261", "transform": "#457B9D",
    "spreading": "#2A9D8F", "rift": "#A8DADC", "intraplate": "#AAAAAA"
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1: Regime distribution by tectonic setting (stacked bar)
ax = axes[0]
regime_names = ["clustered", "ambiguous", "poisson"]
regime_colors = {"clustered": "#E63946", "ambiguous": "#AAAAAA", "poisson": "#457B9D"}
x_labels = []
bar_data = {r: [] for r in regime_names}

for s in setting_order:
    subset = regime_df[regime_df["tectonic_setting"] == s]
    if len(subset) >= 5:
        x_labels.append(f"{s}\n(n={len(subset)})")
        for r in regime_names:
            bar_data[r].append((subset["regime"] == r).sum() / len(subset) * 100)

x = np.arange(len(x_labels))
bottom = np.zeros(len(x_labels))
for r in regime_names:
    vals = bar_data[r]
    ax.bar(x, vals, bottom=bottom, color=regime_colors[r], label=r, alpha=0.8)
    bottom += np.array(vals)

ax.set_xticks(x)
ax.set_xticklabels(x_labels, rotation=30, ha="right")
ax.set_ylabel("Fraction (%)")
ax.set_title("IET Regime by Tectonic Setting")
ax.legend(fontsize=8)

# Panel 2: Weibull k by tectonic setting (box plot)
ax = axes[1]
plot_data = []
plot_labels = []
box_colors = []
for s in setting_order:
    vals = regime_df[regime_df["tectonic_setting"] == s]["weibull_k"]
    if len(vals) >= 5:
        plot_data.append(vals.values)
        plot_labels.append(f"{s}\n(n={len(vals)})")
        box_colors.append(setting_colors[s])

bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, label="Poisson (k=1)")
ax.set_ylabel("Weibull k")
ax.set_title("Temporal Clustering by Tectonic Setting")
ax.tick_params(axis="x", rotation=30)

# Panel 3: Median IET by tectonic setting
ax = axes[2]
medians = []
labels = []
colors = []
for s in setting_order:
    vals = regime_df[regime_df["tectonic_setting"] == s]["median_iet_hours"]
    if len(vals) >= 5:
        medians.append(vals.median())
        labels.append(f"{s}\n(n={len(vals)})")
        colors.append(setting_colors[s])

ax.bar(range(len(medians)), medians, color=colors, alpha=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("Median IET (hours)")
ax.set_title("Median Inter-Event Time by Setting")

fig.suptitle("Inter-Event Time Regime by Tectonic Setting (PB2002)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save_figure(fig, "03_regime_by_tectonic_setting")
plt.show()

# Summary statistics
print("Regime distribution by tectonic setting:")
from scipy.stats import kruskal
for s in setting_order:
    subset = regime_df[regime_df["tectonic_setting"] == s]
    if len(subset) >= 5:
        counts = subset["regime"].value_counts()
        k_med = subset["weibull_k"].median()
        print(f"  {s:15s}: n={len(subset):3d}, k_median={k_med:.3f}, "
              f"clustered={counts.get('clustered', 0)}, "
              f"poisson={counts.get('poisson', 0)}, "
              f"ambiguous={counts.get('ambiguous', 0)}")

# Kruskal-Wallis on Weibull k across settings
groups = [regime_df[regime_df["tectonic_setting"] == s]["weibull_k"].values
          for s in setting_order
          if len(regime_df[regime_df["tectonic_setting"] == s]) >= 5]
H, p = kruskal(*groups)
print(f"\nKruskal-Wallis on Weibull k across settings: H = {H:.1f}, p = {p:.2e}")

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_50119/3519780185.py:56: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,


Regime distribution by tectonic setting:
  subduction     : n=297, k_median=0.496, clustered=276, poisson=4, ambiguous=17
  convergent     : n= 59, k_median=0.427, clustered=53, poisson=1, ambiguous=5
  transform      : n= 91, k_median=0.502, clustered=81, poisson=1, ambiguous=9
  spreading      : n= 62, k_median=0.367, clustered=57, poisson=0, ambiguous=5
  rift           : n= 47, k_median=0.494, clustered=41, poisson=3, ambiguous=3
  intraplate     : n=159, k_median=0.490, clustered=142, poisson=3, ambiguous=14

Kruskal-Wallis on Weibull k across settings: H = 12.2, p = 3.22e-02


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_50119/3519780185.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This notebook classified seismically active 2°×2° grid cells globally into four inter-event time regimes:

- **Poisson-like** (Weibull k ≈ 1): Events arrive randomly and independently, consistent with a memoryless process. Typical of stable tectonic regions with background seismicity.
- **Clustered-like** (Weibull k < 1): Short inter-event times are over-represented, indicating aftershock-dominated or swarm-like behavior.
- **Quasi-periodic-like** (Weibull k > 1): Inter-event times are more regular than random, suggesting stress-loading cycles or repeating earthquake sequences.
- **Ambiguous**: No single distribution fits significantly better than the others.

Key findings:
- The global regime map reveals spatial coherence: subduction zones and active fault systems tend toward clustered behavior, while intraplate regions lean Poisson-like.
- The Weibull k parameter provides a continuous measure of temporal clustering that correlates with tectonic setting.
- Comparison with b-values shows that regions with low b-values (dominated by larger events) tend to exhibit different temporal patterns than high-b-value regions.
- Depth dependence of Weibull k suggests that shallow seismicity is more clustered, consistent with aftershock productivity scaling with depth.
